## Hopfield Network

Today, we are entering the intersection of **linear algebra, dynamical systems, and biology.** We are going to discuss the **Hopfield Network**.

Introduced by John Hopfield in 1982, this model serves as a mathematical description of **associative memory**—the ability of a system to recover a full pattern from a partial or noisy hint. Think of it like hearing the first three notes of a song and your brain "filling in" the rest.

### 1. Mathematical Architecture

A Hopfield network consists of $N$ interconnected units (neurons).

#### The State Space

Each neuron $i$ has a state $s_i \in \{1, -1\}$.

* $+1$ represents a "firing" neuron.
* $-1$ represents a "quiescent" neuron.
The entire state of the network is a vector $\mathbf{s} \in \{-1, 1\}^N$.

#### The Weight Matrix

The "memory" of the network is stored in an $N \times N$ matrix $W$. To ensure the system behaves predictably, we impose two constraints:

1. **Symmetry:** $w_{ij} = w_{ji}$ (The connection from $i$ to $j$ is the same as $j$ to $i$).
2. **No self-loops:** $w_{ii} = 0$ (A neuron doesn't directly influence itself).

#### The Learning Rule (Hebbian Learning)

To store $M$ patterns $\{\mathbf{\xi}^1, \mathbf{\xi}^2, \dots, \mathbf{\xi}^M\}$, we define the weight matrix as:


$$W = \frac{1}{N} \sum_{\mu=1}^M \mathbf{\xi}^\mu (\mathbf{\xi}^\mu)^T$$


*(Note: We then manually set the diagonal elements $w_{ii}$ to 0).*

### 2. Update Rules: How the Network "Thinks"

This is where the dynamics happen. When we give the network a "blurry" input, it needs to update its neurons to find the stored memory.

#### Asynchronous (Sequential) Update

In this mode, we update **one neuron at a time**.

* **The Rule:** Pick a neuron $i$ (at random or in order) and set:

$$s_i \leftarrow \text{sgn}\left( \sum_{j=1}^N w_{ij} s_j \right)$$


* **Stability:** This rule is guaranteed to reach a stable state because it minimizes an **Energy Function**:

$$E = -\frac{1}{2} \sum_{i,j} w_{ij} s_i s_j$$



Every sequential update either decreases $E$ or keeps it the same. It’s like a ball rolling down a hill into a valley.

#### Synchronous (Parallel) Update

In this mode, **all neurons update simultaneously**.

* **The Rule:** $\mathbf{s}(t+1) = \text{sgn}(W \mathbf{s}(t))$.
* **The Risk:** While much faster to compute on modern hardware, this can lead to **limit cycles** (oscillations). Instead of settling in a valley, the system might bounce back and forth between two states forever.

### 3. Explicit Example: A 4-Neuron Network

Let’s store the pattern $\mathbf{\xi} = (1, 1, -1, -1)^T$.

1. **Compute Weights:**

$$W = \begin{pmatrix} 1 \\ 1 \\ -1 \\ -1 \end{pmatrix} \begin{pmatrix} 1 & 1 & -1 & -1 \end{pmatrix} = \begin{pmatrix} 0 & 1 & -1 & -1 \\ 1 & 0 & -1 & -1 \\ -1 & -1 & 0 & 1 \\ -1 & -1 & 1 & 0 \end{pmatrix}$$



*(Diagonals set to 0).*
2. **Test Retrieval:** Suppose we provide a noisy input $\mathbf{s} = (1, -1, -1, -1)^T$ (the second neuron is "wrong").
* **Sequential Update ($s_2$):** The input to $s_2$ is $(1 \cdot 1) + (0 \cdot -1) + (-1 \cdot -1) + (-1 \cdot -1) = 3$.
* Since $3 > 0$, $s_2$ becomes $1$.
* The state is now $(1, 1, -1, -1)^T$, which is our original memory. **Success.**


### 4. Python Implementation

Here is a simple script using `numpy` to demonstrate a Hopfield Network with asynchronous updates.

In [3]:
import numpy as np

def create_weights(patterns):
    """
    Stores patterns using the Hebbian rule.
    """
    num_neurons = patterns.shape[1]
    # W = (1/N) * sum(xi * xi.T)
    W = np.zeros((num_neurons, num_neurons))
    for p in patterns:
        W += np.outer(p, p)
    
    # Enforce zero diagonal (no self-loops)
    np.fill_diagonal(W, 0)
    return W / num_neurons

def update_sequential(state, W, iterations=10):
    """
    Updates neurons one by one in a random order.
    """
    s = state.copy()
    n = len(s)
    for _ in range(iterations):
        # Pick a random order for neurons to update
        indices = np.random.permutation(n)
        for i in indices:
            # s_i = sgn(sum(w_ij * s_j))
            activation = np.dot(W[i], s)
            s[i] = 1 if activation >= 0 else -1
    return s

# --- Example Usage ---

# 1. Define two clear patterns (5 neurons each)
p1 = np.array([1, 1, -1, -1, 1])
p2 = np.array([-1, 1, -1, 1, -1])
all_patterns = np.array([p1, p2])

# 2. Train the network
weights = create_weights(all_patterns)

# 3. Create a noisy version of pattern 1 (flip the first bit)
noisy_input = np.array([-1, 1, -1, -1, 1])

# 4. Recover the memory
recovered = update_sequential(noisy_input, weights)

print(f"Stored Pattern 1: {p1}")
print(f"Noisy Input:     {noisy_input}")
print(f"Recovered:       {recovered}")

Stored Pattern 1: [ 1  1 -1 -1  1]
Noisy Input:     [-1  1 -1 -1  1]
Recovered:       [ 1  1 -1 -1  1]


### Summary for your notes

* **Asynchronous:** More biologically realistic, guaranteed to converge to a local energy minimum.
* **Synchronous:** Mathematically efficient, but can lead to "infinite loops" between two states.
* **Capacity:** A network can only store about $0.14N$ patterns before memories start interfering with each other (crosstalk).